In [1]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls

Mounted at /content/drive
/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
build		    experiments  pyproject.toml  src	     test.sh
build_and_test.sh   LICENSE	 README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	 SECURITY.md	 tests


In [2]:
import os, textwrap

# Where to save logs and models in your Drive
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs_Qwen2-1.5B-Instruct")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_Qwen2-1.5B-Instruct")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

sparsities = [0.0, 0.10, 0.25, 0.4, 0.6]
#datasets = ["wikitext2", "squad", "hotpotqa"]
datasets = ["coqa"]
print("Logs in:", LOG_DIR)
print("Models in:", MODEL_DIR)

Logs in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_Qwen2-1.5B-Instruct
Models in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_Qwen2-1.5B-Instruct


In [3]:
import os
import json
import pandas as pd

# ============================================================
# MODELS: put here the REAL folder names
# ============================================================
models_info = [
    {
        "model": "Qwen/Qwen2-1.5B-Instruct",
        "outer_dir": "Qwen2-1.5B-Instruct",
        "inner_dir": "Qwen-Qwen2-1.5B-Instruct_squad",
    },
    {
        "model": "Qwen/Qwen2-0.5B",
        "outer_dir": "Qwen2-0.5B",
        "inner_dir": "Qwen-Qwen2-0.5B_squad",
    },
    {
        "model": "facebook/opt-125m",
        "outer_dir": "facebook-opt-125m",
        "inner_dir": "facebook-opt-125m_squad",
    },
    {
        "model": "facebook/opt-1.3b",
        "outer_dir": "facebook-opt-1.3b",
        "inner_dir": "facebook-opt-1.3B_squad",
    },
]

benchmarks = {
    "squadv2": {
        "suffix_candidates": ["squadv2"],
        "task_keys": ["squadv2"],
    },
    "coqa": {
        "suffix_candidates": ["coqa"],
        "task_keys": ["coqa"],
    },
    "hotpotqa": {
        "suffix_candidates": ["hotpot", "hotpotqa"],
        "task_keys": ["hotpotqa", "hotpot", None],
    },
}

def get_results_dir(base_eval_dir, outer_dir, inner_dir):
    path = os.path.join(base_eval_dir, outer_dir, inner_dir)
    if os.path.isdir(path):
        return path
    return None

def get_candidate_files(results_dir, suffixes):
    files = []
    for suf in suffixes:
        files.extend([
            os.path.join(results_dir, f"results_s0.00_{suf}.json"),
            os.path.join(results_dir, f"results_ss0.00_{suf}.json"),
            os.path.join(results_dir, f"results_s0p00_{suf}.json"),
            os.path.join(results_dir, f"results_s0p0_{suf}.json"),
            os.path.join(results_dir, f"results_s0_{suf}.json"),
        ])
    return files

def extract_metrics(data, task_keys):
    # custom hotpot style
    if isinstance(data, dict) and "metrics" in data and isinstance(data["metrics"], dict):
        return data["metrics"]

    # lm-eval style
    if isinstance(data, dict) and "results" in data and isinstance(data["results"], dict):
        for k in task_keys:
            if k is not None and k in data["results"]:
                return data["results"][k]

    # top-level task style
    if isinstance(data, dict):
        for k in task_keys:
            if k is not None and k in data and isinstance(data[k], dict):
                return data[k]

    raise KeyError(f"Could not extract metrics. Top-level keys: {list(data.keys())}")

def load_baseline_metrics(results_dir, benchmark_cfg):
    for path in get_candidate_files(results_dir, benchmark_cfg["suffix_candidates"]):
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            metrics = extract_metrics(data, benchmark_cfg["task_keys"])
            metrics = {k: v for k, v in metrics.items() if isinstance(v, (int, float))}
            return path, metrics
    raise FileNotFoundError(f"No baseline file found in {results_dir}")

def pick_metric(metrics, keys):
    for k in keys:
        if k in metrics:
            return metrics[k]
    return None

summary_rows = []

for model_cfg in models_info:
    model_name = model_cfg["model"]
    results_dir = get_results_dir(BASE_EVAL_DIR, model_cfg["outer_dir"], model_cfg["inner_dir"])

    if results_dir is None:
        print(f"[WARN] Missing dir for {model_name}: "
              f"{os.path.join(BASE_EVAL_DIR, model_cfg['outer_dir'], model_cfg['inner_dir'])}")
        continue

    row = {"model": model_name}

    # SQuADv2
    try:
        _, m = load_baseline_metrics(results_dir, benchmarks["squadv2"])
        row["SQuADv2 EM"] = pick_metric(m, ["exact,none", "exact"])
        row["SQuADv2 F1"] = pick_metric(m, ["f1,none", "f1"])
    except Exception as e:
        print(f"[WARN] {model_name} | SQuADv2: {e}")
        row["SQuADv2 EM"] = None
        row["SQuADv2 F1"] = None

    # CoQA
    try:
        _, m = load_baseline_metrics(results_dir, benchmarks["coqa"])
        row["CoQA EM"] = pick_metric(m, ["em,none", "em", "exact"])
        row["CoQA F1"] = pick_metric(m, ["f1,none", "f1"])
    except Exception as e:
        print(f"[WARN] {model_name} | CoQA: {e}")
        row["CoQA EM"] = None
        row["CoQA F1"] = None

    # HotpotQA
    try:
        _, m = load_baseline_metrics(results_dir, benchmarks["hotpotqa"])
        row["HotpotQA EM"] = pick_metric(m, ["em", "exact", "em,none", "exact,none"])
        row["HotpotQA F1"] = pick_metric(m, ["f1", "f1,none"])
    except Exception as e:
        print(f"[WARN] {model_name} | HotpotQA: {e}")
        row["HotpotQA EM"] = None
        row["HotpotQA F1"] = None

    summary_rows.append(row)

baseline_table = pd.DataFrame(summary_rows).round(3)
display(baseline_table)

,model,SQuADv2 EM,SQuADv2 F1,CoQA EM,CoQA F1,HotpotQA EM,HotpotQA F1
0,Qwen/Qwen2-1.5B-Instruct,8.785,15.122,0.600,0.735,0.032,0.077
1,Qwen/Qwen2-0.5B,9.652,18.294,0.456,0.569,0.032,0.073
2,facebook/opt-125m,0.531,2.873,0.169,0.234,0.000,0.015
3,facebook/opt-1.3b,5.458,12.181,0.449,0.572,0.000,0.017


In [4]:
latex_table = baseline_table.copy()
latex_table = latex_table.round(2)
print(latex_table.to_latex(index=False, na_rep="--", float_format="%.2f"))

\begin{tabular}{lrrrrrr}
\toprule
model & SQuADv2 EM & SQuADv2 F1 & CoQA EM & CoQA F1 & HotpotQA EM & HotpotQA F1 \\
\midrule
Qwen/Qwen2-1.5B-Instruct & 8.78 & 15.12 & 0.60 & 0.74 & 0.03 & 0.08 \\
Qwen/Qwen2-0.5B & 9.65 & 18.29 & 0.46 & 0.57 & 0.03 & 0.07 \\
facebook/opt-125m & 0.53 & 2.87 & 0.17 & 0.23 & 0.00 & 0.02 \\
facebook/opt-1.3b & 5.46 & 12.18 & 0.45 & 0.57 & 0.00 & 0.02 \\
\bottomrule
\end{tabular}



In [5]:
latex_table = baseline_table.copy()
latex_table = latex_table.round(2)

print(
    latex_table.to_latex(
        index=False,
        na_rep="--",
        float_format="%.2f",
        caption="Baseline performance of the unpruned models on the considered QA benchmarks.",
        label="tab:baseline_results"
    )
)

\begin{table}
\caption{Baseline performance of the unpruned models on the considered QA benchmarks.}
\label{tab:baseline_results}
\begin{tabular}{lrrrrrr}
\toprule
model & SQuADv2 EM & SQuADv2 F1 & CoQA EM & CoQA F1 & HotpotQA EM & HotpotQA F1 \\
\midrule
Qwen/Qwen2-1.5B-Instruct & 8.78 & 15.12 & 0.60 & 0.74 & 0.03 & 0.08 \\
Qwen/Qwen2-0.5B & 9.65 & 18.29 & 0.46 & 0.57 & 0.03 & 0.07 \\
facebook/opt-125m & 0.53 & 2.87 & 0.17 & 0.23 & 0.00 & 0.02 \\
facebook/opt-1.3b & 5.46 & 12.18 & 0.45 & 0.57 & 0.00 & 0.02 \\
\bottomrule
\end{tabular}
\end{table}



In [6]:
latex_table = baseline_table.copy()

latex_table["model"] = latex_table["model"].replace({
    "Qwen/Qwen2-1.5B-Instruct": "Qwen2-1.5B-Instruct",
    "Qwen/Qwen2-0.5B": "Qwen2-0.5B",
    "facebook/opt-125m": "OPT-125M",
    "facebook/opt-1.3b": "OPT-1.3B",
})

latex_table = latex_table.round(2)

print(
    latex_table.to_latex(
        index=False,
        na_rep="--",
        float_format="%.2f",
        caption="Baseline performance of the unpruned models on the considered QA benchmarks.",
        label="tab:baseline_results"
    )
)

\begin{table}
\caption{Baseline performance of the unpruned models on the considered QA benchmarks.}
\label{tab:baseline_results}
\begin{tabular}{lrrrrrr}
\toprule
model & SQuADv2 EM & SQuADv2 F1 & CoQA EM & CoQA F1 & HotpotQA EM & HotpotQA F1 \\
\midrule
Qwen2-1.5B-Instruct & 8.78 & 15.12 & 0.60 & 0.74 & 0.03 & 0.08 \\
Qwen2-0.5B & 9.65 & 18.29 & 0.46 & 0.57 & 0.03 & 0.07 \\
OPT-125M & 0.53 & 2.87 & 0.17 & 0.23 & 0.00 & 0.02 \\
OPT-1.3B & 5.46 & 12.18 & 0.45 & 0.57 & 0.00 & 0.02 \\
\bottomrule
\end{tabular}
\end{table}



In [ ]:
def get_metric(metrics, keys):
    for k in keys:
        if k in metrics:
            return metrics[k]
    return None

pretty_rows = []

for model_cfg in models_info:
    model_hf = model_cfg["model_hf"]
    base_subdir = model_cfg["base_subdir"]
    calib_folder = model_cfg["calib_folder"]

    results_dir = find_results_dir(model_hf, base_subdir, calib_folder)
    if results_dir is None:
        continue

    row = {"model": model_hf}

    # SQuADv2
    try:
        _, m = load_baseline_metrics(results_dir, "squadv2", benchmarks["squadv2"])
        row["SQuADv2 EM"] = get_metric(m, ["exact,none", "exact"])
        row["SQuADv2 F1"] = get_metric(m, ["f1,none", "f1"])
    except:
        row["SQuADv2 EM"] = None
        row["SQuADv2 F1"] = None

    # CoQA
    try:
        _, m = load_baseline_metrics(results_dir, "coqa", benchmarks["coqa"])
        row["CoQA F1"] = get_metric(m, ["f1,none", "f1"])
    except:
        row["CoQA F1"] = None

    # HotpotQA
    try:
        _, m = load_baseline_metrics(results_dir, "hotpotqa", benchmarks["hotpotqa"])
        row["HotpotQA EM"] = get_metric(m, ["em", "exact", "em,none", "exact,none"])
        row["HotpotQA F1"] = get_metric(m, ["f1", "f1,none"])
    except:
        row["HotpotQA EM"] = None
        row["HotpotQA F1"] = None

    pretty_rows.append(row)

pretty_table = pd.DataFrame(pretty_rows).round(2)
display(pretty_table)

,model,SQuADv2 EM,SQuADv2 F1,CoQA F1,HotpotQA EM,HotpotQA F1
0,facebook/opt-125m,0.53,2.87,0.23,0.0,0.02
